## Ensembles for Customer Satisfaction Prediction

KATE expects your code to define variables with specific names that correspond to certain things we are interested in.

KATE will run your notebook from top to bottom and check the latest value of those variables, so make sure you don't overwrite them.

* Remember to uncomment the line assigning the variable to your answer and don't change the variable or function names.
* Use copies of the original or previous DataFrames to make sure you do not overwrite them by mistake.

You will find instructions below about how to define each variable.

Once you're happy with your code, upload your notebook to KATE to check your feedback.

Businesses can improve their services by tailoring them to individual customers. One important factor is knowing when customers are dissatisfied. Based on their records, one can use machine learning tools to make predictions about which customers are more at risk of being dissatisfied than others. Such predictions allow for individualised actions that may help retain customers and will improve quality.

In this assignment, we will build a prediction model for bank account owners' satisfaction. The record includes more than 300 features for each client, including variables related to their balance and which banking operations they have performed. Many of these variables are sparse; some numerical, some categorical. 

Ensemble methods based on decision trees, such as random forests and boosting algorithms, have been very successful in modeling such heterogeneous tabular data. To learn how these models work, you will implement them step-by-step, and see how the performance of your predictions improve.

### Load the data

Load the data in `data/train_data.csv` with `pandas`. Inspect its content with `.head()`, `.shape` and other methods of your choice.

In [23]:

import pandas as pd

# Load the training data
train_df = pd.read_csv('data/train_data.csv')


print(train_df.head())




In [24]:
# Show the shape of the DataFrame
print("Shape:", train_df.shape)

In [25]:
# Show column names and data types
print(train_df.info())

#### Target variable

The last column, named `TARGET`, is the variable to be predicted. `TARGET=1` represents a dissatisfied customer. Inspect the target column with `.value_counts()`. 

What is the proportion of dissatisfied customers? Is the dataset balanced or imbalanced?

In [26]:
# Check the distribution of the target variable
print(train_df['TARGET'].value_counts())
print(train_df['TARGET'].value_counts(normalize=True))


Separate the data into features `X` and target `y`. Split the data into `_train` and `_val` sets, using the default sklearn ratios and stratifying on `y` to keep the same level of imbalance.

*Hint: you may use `train_test_split()` for stratified splits.*

In [27]:

# Drop the index column
train_df = train_df.drop(columns=['Unnamed: 0'])

# Separate features and target
X = train_df.drop(columns=['TARGET']).copy()
y = train_df['TARGET'].copy()

from sklearn.model_selection import train_test_split

# Stratified split to maintain class imbalance in both sets
X_train, X_val, y_train, y_val = train_test_split(
    X, y, 
    test_size=0.25,         # Default is 0.25
    random_state=42,        # For reproducibility
    stratify=y              # Maintain class proportions
)




In [28]:
print("Train set shape:", X_train.shape)
print("Validation set shape:", X_val.shape)
print("Proportion of dissatisfied customers in train:", y_train.mean())
print("Proportion of dissatisfied customers in validation:", y_val.mean())


### Basic modelling pipeline

Implement a basic modelling pipeline for a Decision Tree Classifier, fitting the training data and printing the training and validation accuracy.

In [29]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score

# Initialize the model
dt_model = DecisionTreeClassifier(random_state=42)

# Fit the model on the training data
dt_model.fit(X_train, y_train)

# Predict on training and validation sets
y_train_pred = dt_model.predict(X_train)
y_val_pred = dt_model.predict(X_val)

# Calculate accuracy
train_accuracy = accuracy_score(y_train, y_train_pred)
val_accuracy = accuracy_score(y_val, y_val_pred)

print("Training accuracy:", train_accuracy)
print("Validation accuracy:", val_accuracy)


Note that the accuracy score is quite high, even for this very simple model. Take a moment to think why this high score is not that significant.

#### ROC curve metric

Change your scoring metric to `roc_auc_score`, which calculates the area below the ROC curve of your **prediction probabilities**, instead of using the binary prediction decisions.

*Hint: `model.predict_proba(X)` will return the probabilities for `y = 0` and `y = 1`, slice this to only return the `y = 1` predictions. 

In [30]:
from sklearn.metrics import roc_auc_score

# Get prediction probabilities for both training and validation sets
y_train_proba = dt_model.predict_proba(X_train)[:, 1]  # Get probabilities for y=1 only
y_val_proba = dt_model.predict_proba(X_val)[:, 1]     # Get probabilities for y=1 only

# Calculate ROC AUC scores
train_roc_auc = roc_auc_score(y_train, y_train_proba)
val_roc_auc = roc_auc_score(y_val, y_val_proba)

print("Training ROC AUC:", train_roc_auc)
print("Validation ROC AUC:", val_roc_auc)

#### Baseline score for random predictions

Calculate the ROC AUC for random uniform prediction probabilities. 

Is the Decision Tree better? Based on the training and validation scores, what is the problem with the Decision Tree model?

*Hint: You can use `np.random.uniform` to create an array of random numbers under 1.*

In [31]:
import numpy as np
from sklearn.metrics import roc_auc_score

# Generate random uniform prediction probabilities
np.random.seed(42)  # For reproducibility
random_proba = np.random.uniform(0, 1, len(y_val))

# Calculate ROC AUC for random predictions
random_roc_auc = roc_auc_score(y_val, random_proba)

print("Random baseline ROC AUC:", random_roc_auc)
print("Decision Tree training ROC AUC:", train_roc_auc)
print("Decision Tree validation ROC AUC:", val_roc_auc)

# Compare performance
print(f"\nIs Decision Tree better than random? {val_roc_auc > random_roc_auc}")
print(f"Decision Tree improvement over random: {val_roc_auc - random_roc_auc:.3f}")


Create a function named `test_model(model, X_train, y_train, X_val, y_val)` that performs the basic prediction pipeline, receiving as argument the model and data, fitting the training data, and returning the training and validation `roc_auc_score`. Check that it works with the Decision Tree model.

In [32]:
from sklearn.metrics import roc_auc_score

def test_model(model, X_train, y_train, X_val, y_val):
    # Fit the model
    model.fit(X_train, y_train)
    # Predict probabilities for the positive class
    y_train_proba = model.predict_proba(X_train)[:, 1]
    y_val_proba = model.predict_proba(X_val)[:, 1]
    # Calculate ROC AUC scores
    train_roc_auc = roc_auc_score(y_train, y_train_proba)
    val_roc_auc = roc_auc_score(y_val, y_val_proba)
    return train_roc_auc, val_roc_auc


train_roc_auc, val_roc_auc = test_model(dt_model, X_train, y_train, X_val, y_val)

print("Decision Tree training ROC AUC:", train_roc_auc)
print("Decision Tree validation ROC AUC:", val_roc_auc)


## Optimising decision trees 

Optimise your model by changing the `max_depth` hyperparameter. Iterate through and calculate the training and validation scores for depths of `[1,2,5,10,20,50]`. Which depth has the best performance? Remember that we want a model that performs well on both the training and validation datasets. 

In [33]:
depths = [1, 2, 5, 10, 20, 50]
results = []

for depth in depths:
    model = DecisionTreeClassifier(max_depth=depth, random_state=42)
    train_roc_auc, val_roc_auc = test_model(model, X_train, y_train, X_val, y_val)
    results.append((depth, train_roc_auc, val_roc_auc))
    print(f"Depth: {depth} | Train ROC AUC: {train_roc_auc:.3f} | Val ROC AUC: {val_roc_auc:.3f}")


To evaluate your models, we will test your data on an unseen testing set. Load the test at `data/test_data.csv`.

In [34]:
test_data = pd.read_csv('data/test_data.csv')

print(test_data.head())
print("Shape:", test_data.shape)
print(test_data.info())

 Calculate the prediction probabilities for the `test_data` using the best Decision Tree depth, saving them in a variable named `dtc_preds`. `dtc_preds` should be an numpy array containing the probabilities for `y = 1` only.

In [35]:
from sklearn.tree import DecisionTreeClassifier

# Re-train if needed. Best decision tree is depth = 5
dtc_best = DecisionTreeClassifier(max_depth=5, random_state=42)
dtc_best.fit(X_train, y_train)

import numpy as np

dtc_preds = dtc_best.predict_proba(test_data)[:, 1]
print(dtc_preds[:10])  # Show first 10 probabilities
print(type(dtc_preds)) # Should be <class 'numpy.ndarray'>


### Bagging

While decision trees are prone to overfitting, their ensemble can be a powerful predictor. A bagging ensemble of decision trees uses the average prediction of multiple decision trees base models, that have each been trained with on a different set of data samples. This resampling processes is known as Bootstraping. (The name Bagging comes from Bootstrap + Aggregation, meaning bootstrap your data and then aggregate your model outputs.)

You will create a Bagging model class, named `myBagging`, filling in the class structure below.

The `.fit()` method should fit each base model with a bootstrap sample of the data (with replacement), with data size proportional by the hyperparameter `subsample`. That is, if `subsample=0.5`, each base model should get half the total number of samples.

The `.predict_proba()` method should estimate and average the prediction probabilities of the base models.

*Hint: You can use the `resample()` function for creating bootstrap samples.*

In [36]:
from sklearn.base import clone

class myBagging:
    def __init__(self, base_models, subsample = 1.):
        self.n_models = len(base_models)
        self.base_models = base_models
        self.subsample = subsample
        
    def fit(self, X, y):
        '''Loop over base models, generate a bootstrap sample of the data with 'resample()',
           and fit them to the data.
           
           To access the variables inside the myBagging class, use the 'self.' prefix, 
           i.e. self.base_models, self.n_models and self.subsample
        '''
        


        from sklearn.utils import resample
        self.fitted_models = []
        n_samples = int(self.subsample * len(X))
        for model in self.base_models:
            # Bootstrap sample
            X_resampled, y_resampled = resample(
                X, y, 
                n_samples=n_samples, 
                replace=True
            )
            model_clone = clone(model)
            model_clone.fit(X_resampled, y_resampled)
            self.fitted_models.append(model_clone)
        pass
    
    def predict_proba(self, X):
        '''Return the ensemble predictions, given by the average prediction probability over base models.
           It should be an array with the length of the dataset.'''
        

        # Collect predictions from each model
        probas = np.zeros((X.shape[0], 2))
        for model in self.fitted_models:
            probas += model.predict_proba(X)
        probas /= self.n_models
        return probas

        pass
    


Run and score a Bagging ensemble, with 10 decision trees as base learners, with maximum depth 10 and subsample 0.5. Use your `myBagging` class and `test_model()`.

In [37]:
from sklearn.tree import DecisionTreeClassifier

base_models = [DecisionTreeClassifier(max_depth=10, random_state=i) for i in range(10)]
bagging_model = myBagging(base_models=base_models, subsample=0.5)
train_roc_auc, val_roc_auc = test_model(bagging_model, X_train, y_train, X_val, y_val)
print("Bagging training ROC AUC:", train_roc_auc)
print("Bagging validation ROC AUC:", val_roc_auc)


### Sklearn comparison

For comparison, run the below cell to see how the `sklearn` implementation, `BaggingClassifier` does. 

Is there much of a difference? Why could that be?

In [38]:
from sklearn.ensemble import BaggingClassifier

test_model(BaggingClassifier(estimator=DecisionTreeClassifier(max_depth=10), 
                              n_estimators=10, max_samples=0.5), X_train, y_train, X_val, y_val)

### Random Forests

Random Forests are bagged ensembles of decision trees in which, at each node split during training, only a fraction of the features are considered for the optimal split (e.g. for optimal Gini gain).

Run and score a Random Forest version of your `myBagging` classifier, by changing the `max_features` parameter inside the base learner.

In [39]:
from sklearn.tree import DecisionTreeClassifier

# 'sqrt' is a common choice for Random Forests in classification
base_models_rf = [
    DecisionTreeClassifier(max_depth=10, max_features='sqrt', random_state=i)
    for i in range(10)
]
rf_bagging_model = myBagging(base_models=base_models_rf, subsample=0.5)
train_roc_auc_rf, val_roc_auc_rf = test_model(rf_bagging_model, X_train, y_train, X_val, y_val)
print("Random Forest (Bagging) training ROC AUC:", train_roc_auc_rf)
print("Random Forest (Bagging) validation ROC AUC:", val_roc_auc_rf)


### Sklearn comparison

For comparison, run and score the `sklearn` implementation, `RandomForestClassifier`.

Is there much of a difference? Why could that be?

In [40]:
from sklearn.ensemble import RandomForestClassifier

rf = RandomForestClassifier(
    n_estimators=10,
    max_depth=10,
    max_features='sqrt',
    random_state=42
)
train_roc_auc_rf, val_roc_auc_rf = test_model(rf, X_train, y_train, X_val, y_val)
print("sklearn Random Forest training ROC AUC:", train_roc_auc_rf)
print("sklearn Random Forest validation ROC AUC:", val_roc_auc_rf)


### Optimise your Random Forest

Optimise your Random Forest hyperparameters, both for the myBagging class and for the base decision trees. 

Calculate the prediction probabilities for the `test_data` using your best random forest, saving them in a variable named `rf_preds`. Like before `rf_preds` should be an numpy array containing the probabilities for `y = 1` only.


In [41]:
# Quick optimization with fewer combinations
print("Quick Random Forest optimization...")

# Test a few promising combinations
param_combinations = [
    {'n_est': 20, 'depth': 10, 'features': 'sqrt', 'subsample': 0.8},
    {'n_est': 30, 'depth': 15, 'features': 'sqrt', 'subsample': 0.8},
    {'n_est': 50, 'depth': 10, 'features': 'sqrt', 'subsample': 1.0},
    {'n_est': 20, 'depth': 20, 'features': 0.5, 'subsample': 0.8},
    {'n_est': 30, 'depth': 10, 'features': 'log2', 'subsample': 0.8},
]

best_score = 0
best_params = {}

for params in param_combinations:
    base_models_rf = [
        DecisionTreeClassifier(
            max_depth=params['depth'], 
            max_features=params['features'], 
            random_state=i
        )
        for i in range(params['n_est'])
    ]
    
    rf_model = myBagging(base_models=base_models_rf, subsample=params['subsample'])
    train_roc_auc, val_roc_auc = test_model(rf_model, X_train, y_train, X_val, y_val)
    
    print(f"Params: {params} | Val ROC AUC: {val_roc_auc:.4f}")
    
    if val_roc_auc > best_score:
        best_score = val_roc_auc
        best_params = params

# Train final model and generate predictions
final_base_models = [
    DecisionTreeClassifier(
        max_depth=best_params['depth'],
        max_features=best_params['features'],
        random_state=i
    )
    for i in range(best_params['n_est'])
]

best_rf_model = myBagging(base_models=final_base_models, subsample=best_params['subsample'])
best_rf_model.fit(X_train, y_train)

# Generate rf_preds
rf_preds = best_rf_model.predict_proba(test_data)[:, 1]

Note that including more decision trees improve performance but increases the computational cost of training linearly. The `max_depth` and `max_features` arguments can heavily cut the training time, by reducing the tree size and number of features considered at each split.

## Gradient Boosting

We will now implement a more sophisticated ensemble, Gradient Boosting, in which the base models are trained sequentially. Each new base model predicts what previous base models missed. 

As gradient boosting requires a continuous gradient, it can only use regression models for the base learner. 

For this exercise, we will perform regression directly on the 0-1 class labels, and treat the raw outputs as probabilities. 

We will try to setup the base models to optimise the MSE loss function against the class-labels, for which the gradient becomes simply the residual errors. 

When applied to probabilities, the MSE is known as the Brier score. 

Whilst performing this exercise, have a think about whether this is a robust approach. 

If not, what would you change either to your base-learners, meta-algorithm, or evaluation metrics to make this more robust?

In the below structure, fill the `.fit()` and `.predict_proba()` functions. When returning probabilities remember that you will need to return an array containing the probabilities for `y = 0` and `y = 1`. 

In [45]:
from sklearn.base import clone
import numpy as np

class myGradientBoosting:
    
    def __init__(self, base_models, learning_rate=0.5):
        self.n_models = len(base_models)
        self.models = base_models
        self.learning_rate = learning_rate
        self.fitted_models = []
    
    def fit(self, x, y):
        ''' The `.fit()` function should loop over each base model 
         fitting it to the residual of the ensemble predictions so far, for the MSE loss:
         
         predictions = 0
         for each base model:
             residual = y - predictions   
             fit base model and make new predictions
             predictions = predictions + learning_rate * new_prediction 
        '''
        
        # Initialize predictions to zero
        predictions = np.zeros(len(y))
        self.fitted_models = []
        
        # Loop over each base model
        for model in self.models:
            # Calculate residuals (what the current ensemble is missing)
            residual = y - predictions
            
            # Clone the model to avoid modifying the original
            model_clone = clone(model)
            
            # Fit the model to predict the residuals
            model_clone.fit(x, residual)
            
            # Get predictions from this model
            new_prediction = model_clone.predict(x)
            
            # Update ensemble predictions with learning rate
            predictions = predictions + self.learning_rate * new_prediction
            
            # Store the fitted model
            self.fitted_models.append(model_clone)

            
    def predict_proba(self, x):
        ''' Generate the ensemble prediction, by looping over each base model.
            Get their predictions and sum them, scaled by the learning rate.
        
            Trick: Regressor models return only one prediction (instead of two probabilities in the Classifiers).
                   To make your class compatible with test_model(), you need to output the probabilities for both
                   y = 0 and y = 1, this can be done using np.array([1-probs, probs]).T'''
        
        # Initialize predictions to zero
        predictions = np.zeros(len(x))
        
        # Loop over each fitted model and accumulate predictions
        for model in self.fitted_models:
            model_prediction = model.predict(x)
            predictions = predictions + self.learning_rate * model_prediction
        
        # Clip predictions to [0, 1] range to ensure valid probabilities
        predictions = np.clip(predictions, 0, 1)
        
        # Convert to probability format: [prob_class_0, prob_class_1]
        probs_class_1 = predictions
        probs_class_0 = 1 - probs_class_1
        
        # Return as required format: np.array([1-probs, probs]).T
        return np.array([probs_class_0, probs_class_1]).T

Run and score a Gradient Boosting model, with 20 regression decision trees as your base learners, with maximum depth 5, maximum feature 0.5, and learning rate 0.5. Use your `myGradientBoosting` class and `test_model()`. 

In [46]:
from sklearn.tree import DecisionTreeRegressor

# Create 20 regression decision trees as base learners
base_models_gb = [
    DecisionTreeRegressor(
        max_depth=5, 
        max_features=0.5, 
        random_state=i
    ) 
    for i in range(20)
]

# Create the gradient boosting model with learning rate 0.5
gb_model = myGradientBoosting(base_models=base_models_gb, learning_rate=0.5)

# Test the model using the test_model function
train_roc_auc_gb, val_roc_auc_gb = test_model(gb_model, X_train, y_train, X_val, y_val)

print("Gradient Boosting Results:")
print("=" * 30)
print(f"Training ROC AUC: {train_roc_auc_gb:.4f}")
print(f"Validation ROC AUC: {val_roc_auc_gb:.4f}")
print(f"Difference (overfitting check): {train_roc_auc_gb - val_roc_auc_gb:.4f}")

### Sklearn comparison

For comparison, run and score the `sklearn` implementation, `GradientBoostingClassifier` in the cell below.

In [47]:
from sklearn.ensemble import GradientBoostingClassifier

# Create sklearn's GradientBoostingClassifier with similar parameters
sklearn_gb = GradientBoostingClassifier(
    n_estimators=20,        # 20 base learners
    max_depth=5,           # maximum depth 5
    max_features=0.5,      # maximum features 0.5
    learning_rate=0.5,     # learning rate 0.5
    random_state=42        # for reproducibility
)

# Test the sklearn model
train_roc_auc_sklearn, val_roc_auc_sklearn = test_model(sklearn_gb, X_train, y_train, X_val, y_val)

print("Sklearn GradientBoostingClassifier Results:")
print("=" * 45)
print(f"Training ROC AUC: {train_roc_auc_sklearn:.4f}")
print(f"Validation ROC AUC: {val_roc_auc_sklearn:.4f}")
print(f"Difference (overfitting check): {train_roc_auc_sklearn - val_roc_auc_sklearn:.4f}")

print("\nComparison with myGradientBoosting:")
print("=" * 35)
print(f"Custom implementation - Val ROC AUC: {val_roc_auc_gb:.4f}")
print(f"Sklearn implementation - Val ROC AUC: {val_roc_auc_sklearn:.4f}")
print(f"Performance difference: {val_roc_auc_sklearn - val_roc_auc_gb:.4f}")

### Optimise your Gradient Boosting 

Optimise your myGradientBoosting and decision tree hyperparameters.

Calculate the prediction probabilities for the `test_data` using your best gradient boosting model saving them in a variable named `gb_preds`. Like before `gb_preds` should be an numpy array containing the probabilities for `y = 1` only.


In [49]:
# Quick optimization with promising combinations
print("Quick Gradient Boosting optimization...")

param_combinations = [
    {'n_est': 20, 'depth': 5, 'features': 0.5, 'lr': 0.3},
    {'n_est': 30, 'depth': 3, 'features': 0.5, 'lr': 0.5},
    {'n_est': 15, 'depth': 7, 'features': 0.3, 'lr': 0.3},
    {'n_est': 25, 'depth': 5, 'features': 'sqrt', 'lr': 0.1},
    {'n_est': 20, 'depth': 3, 'features': 0.7, 'lr': 0.5},
    {'n_est': 10, 'depth': 10, 'features': 0.5, 'lr': 0.7},
]

best_score = 0
best_params = {}

for params in param_combinations:
    base_models_gb = [
        DecisionTreeRegressor(
            max_depth=params['depth'],
            max_features=params['features'],
            random_state=i
        )
        for i in range(params['n_est'])
    ]
    
    gb_model = myGradientBoosting(base_models=base_models_gb, learning_rate=params['lr'])
    
    try:
        train_roc_auc, val_roc_auc = test_model(gb_model, X_train, y_train, X_val, y_val)
        print(f"Params: {params} | Val ROC AUC: {val_roc_auc:.4f}")
        
        if val_roc_auc > best_score:
            best_score = val_roc_auc
            best_params = params
            
    except Exception as e:
        print(f"Error with params {params}: {e}")
        continue

print(f"\nBest params: {best_params}")
print(f"Best validation ROC AUC: {best_score:.4f}")

# Train the final optimized model
print("\nTraining final optimized Gradient Boosting model...")

final_base_models = [
    DecisionTreeRegressor(
        max_depth=best_params['depth'] if 'depth' in best_params else best_params['max_depth'],
        max_features=best_params['features'] if 'features' in best_params else best_params['max_features'],
        random_state=i
    )
    for i in range(best_params['n_est'] if 'n_est' in best_params else best_params['n_estimators'])
]

best_gb_model = myGradientBoosting(
    base_models=final_base_models,
    learning_rate=best_params['lr'] if 'lr' in best_params else best_params['learning_rate']
)

# Fit the best model on training data
best_gb_model.fit(X_train, y_train)

# Generate predictions on test data
gb_preds = best_gb_model.predict_proba(test_data)[:, 1]

print(f"Generated {len(gb_preds)} predictions for test data")
print(f"gb_preds type: {type(gb_preds)}")
print(f"gb_preds shape: {gb_preds.shape}")
print(f"Sample predictions: {gb_preds[:5]}")
print(f"Min prediction: {gb_preds.min():.4f}")
print(f"Max prediction: {gb_preds.max():.4f}")

# Verify final model performance
final_train_roc, final_val_roc = test_model(best_gb_model, X_train, y_train, X_val, y_val)
print(f"\nFinal model performance:")
print(f"Training ROC AUC: {final_train_roc:.4f}")
print(f"Validation ROC AUC: {final_val_roc:.4f}")
print(f"Overfitting check: {final_train_roc - final_val_roc:.4f}")

Try to think about the difference between your implementation and the GradientBoostingClassifier.

Are there any fundamental differences? If so, why?

You could try looking at the distribution of your output probabilities for each model.